In [0]:
import json
import random
import shutil
import time
from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

TORONTO = ZoneInfo("America/Toronto")

dbutils.widgets.text("catalog", "adb_fgfworkshop")
dbutils.widgets.text("team", "team7")
dbutils.widgets.text("as_of_date", datetime.now(TORONTO).strftime("%Y-%m-%d"))
dbutils.widgets.dropdown("build_pipeline", "true", ["true", "false"])

catalog = dbutils.widgets.get("catalog")
team = dbutils.widgets.get("team")
as_of = datetime.strptime(dbutils.widgets.get("as_of_date"), "%Y-%m-%d").date()
landing = Path(f"/Volumes/{catalog}/{team}/landing")
random.seed(f"{team}-{as_of}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{team}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{team}.landing")
for child in landing.iterdir():
    shutil.rmtree(child) if child.is_dir() else child.unlink()

BRANDS = ["Stonefire", "ACE Bakery", "Wonder", "D'Italiano", "Casa Mendosa", "Gadoua", "FGF Foodservice"]
BRAND_WEIGHTS = [18, 17, 17, 16, 11, 11, 10]
RETAILERS = ["Costco", "Walmart", "Loblaw", "Sobeys", "Metro", "Save-On-Foods"]
REGIONS = ["ON", "QC", "BC", "AB", "Atlantic"]
HOUR_WEIGHTS = [1, 1, 1, 1, 1, 2, 5, 10, 11, 11, 10, 7, 5, 4, 4, 4, 3, 3, 2, 2, 2, 1, 1, 1]
FIRST = ["Ava", "Noah", "Liam", "Emma", "Olivia", "Lucas", "Mia", "Ethan", "Amelia", "Sophie", "Marc", "Chloe"]
LAST = ["Patel", "Chen", "Martin", "Singh", "Brown", "Nguyen", "Tremblay", "Roy", "Wilson", "Gagnon", "Clark"]


def month_start(d: date, months_back: int) -> date:
    y, m = d.year, d.month - months_back
    while m <= 0:
        m, y = m + 12, y - 1
    return date(y, m, 1)


def order_ts(day: date, hour: int | None = None) -> str:
    hour = random.choices(range(24), weights=HOUR_WEIGHTS)[0] if hour is None else hour
    ts = datetime(day.year, day.month, day.day, hour, random.randint(0, 59), random.randint(0, 59), tzinfo=TORONTO)
    now = datetime.now(TORONTO)
    if ts > now:
        ts = max(now - timedelta(minutes=random.randint(1, 90)), datetime(day.year, day.month, day.day, tzinfo=TORONTO))
    return ts.isoformat()


def write_json_lines(folder: str, rows: list[dict]) -> None:
    (landing / folder).mkdir(parents=True, exist_ok=True)
    with open(landing / folder / "seed.json", "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, default=str) + "\n")


In [0]:
products, skus_by_brand = [], defaultdict(list)
for brand in BRANDS:
    code = brand.upper().replace(" ", "").replace("'", "")[:8]
    for i in range(1, 13 if brand == "FGF Foodservice" else 9):
        list_price = round(random.uniform(3.25, 9.75), 2)
        products.append({
            "sku": f"{code}-{i:03d}",
            "brand": brand,
            "category": random.choice(["Bread", "Buns", "Flatbread", "Rolls", "Bagels", "Tortillas"]),
            "pack_size": random.choice(["6 ct", "8 ct", "12 ct", "18 ct"]),
            "list_price": list_price,
            "standard_cost": round(list_price * random.uniform(0.45, 0.68), 2),
        })
        skus_by_brand[brand].append(f"{code}-{i:03d}")
product = {p["sku"]: p for p in products}

customers = []
for idx in range(1, 201):
    if idx <= 4:
        retailer, region = "Costco", random.choice(["ON", "BC", "AB", "Atlantic"])
    elif idx <= 8:
        retailer, region = "Costco", "QC"
    else:
        retailer = random.choices(RETAILERS, weights=[18, 40, 42, 34, 34, 26])[0]
        region = random.choices(REGIONS, weights=[40, 24, 16, 10, 10])[0]
    first, last = random.choice(FIRST), random.choice(LAST)
    customers.append({
        "ship_to_cust": f"C{idx:04d}",
        "customer_name": f"{retailer} {region} #{idx:03d}",
        "retailer": retailer,
        "region": region,
        "channel": {"Costco": "Club", "Walmart": "Mass"}.get(retailer, "Grocery"),
        "buyer_name": f"{first} {last}",
        "buyer_email": f"{first.lower()}.{last.lower()}{idx}@{retailer.lower().replace('-', '').replace(' ', '')}.example.com",
        "buyer_phone": f"+1-{random.choice(['416', '514', '604', '403', '902'])}-555-{random.randint(1000, 9999)}",
    })
hotspots = [c["ship_to_cust"] for c in customers[:4]]
random.shuffle(customers)

promo_start, promo_end = as_of - timedelta(days=1), as_of + timedelta(days=7)
ACE, WONDER, CASA = "PROMO-ACE-COSTCO-TPD", "PROMO-WONDER-WALMART-ENDCAP", "PROMO-CASA-SOBEYS-FLYER"
promotions = [
    {"promo_id": ACE, "brand": "ACE Bakery", "sku": None, "retailer": "Costco", "promo_type": "TPD", "discount_pct": 24.0},
    {"promo_id": WONDER, "brand": "Wonder", "sku": None, "retailer": "Walmart", "promo_type": "END_CAP", "discount_pct": 9.0},
    {"promo_id": CASA, "brand": "Casa Mendosa", "sku": None, "retailer": "Sobeys", "promo_type": "FLYER", "discount_pct": 12.0},
]
for p in promotions:
    p.update(start_date=promo_start, end_date=promo_end)
for idx in range(4, 21):
    brand = random.choice(BRANDS)
    start = as_of - timedelta(days=random.randint(30, 80))
    promotions.append({
        "promo_id": f"PROMO-{idx:03d}",
        "brand": brand,
        "sku": random.choice(skus_by_brand[brand]) if random.random() < 0.5 else None,
        "retailer": random.choice(RETAILERS) if random.random() < 0.7 else None,
        "promo_type": random.choice(["TPD", "END_CAP", "FLYER"]),
        "start_date": start,
        "end_date": start + timedelta(days=random.randint(7, 21)),
        "discount_pct": round(random.uniform(5, 18), 1),
    })


In [0]:
headers, lines = [], []
actuals = defaultdict(lambda: [0.0, 0])
decline_sku = skus_by_brand["Stonefire"][0]


def add_order(cust: dict, day: date, status: str, rdd: date, items: list[dict], hour: int | None = None) -> None:
    order_id = f"SO-{len(headers) + 1:06d}"
    shipped = status in ("SHIPPED", "INVOICED")
    headers.append({
        "order_id": order_id,
        "order_ts": order_ts(day, hour),
        "ship_to_cust": cust["ship_to_cust"],
        "status": status,
        "requested_delivery_date": rdd,
        "actual_ship_date": rdd + timedelta(days=random.randint(-1, 1)) if shipped else None,
    })
    for n, item in enumerate(items, start=1):
        lines.append({"order_id": order_id, "line_no": n, **item})
        if status != "CANCELLED":
            key = (product[item["sku"]]["brand"], cust["retailer"], month_start(day, 0))
            actuals[key][0] += item["unit_price"] * item["qty"] - item["discount_amt"]
            actuals[key][1] += item["qty"]


def make_line(cust: dict, day: date) -> dict:
    retailer, days_ago = cust["retailer"], (as_of - day).days
    brand = random.choices(BRANDS, weights=BRAND_WEIGHTS)[0]
    sku = random.choice([s for s in skus_by_brand[brand] if not (retailer == "Loblaw" and s == decline_sku)])
    lp, cost = product[sku]["list_price"], product[sku]["standard_cost"]
    qty, price, promo = random.randint(6, 48), lp * random.uniform(0.88, 1.0), None
    in_promo = promo_start <= day <= as_of
    if in_promo and brand == "ACE Bakery" and retailer == "Costco":
        promo, qty = ACE, int(qty * 1.3)
        price = cost * random.uniform(0.80, 0.95) if cust["ship_to_cust"] in hotspots else lp * 0.76
    if in_promo and brand == "Wonder" and retailer == "Walmart":
        promo, qty = WONDER, max(4, int(qty * 0.8))
    if in_promo and brand == "Casa Mendosa" and retailer == "Sobeys":
        promo, qty, price = CASA, int(qty * 1.45), lp * 0.88
    if brand == "Wonder" and days_ago <= 27 and retailer in ("Costco", "Walmart"):
        target_margin = random.uniform(0.23, 0.25) if retailer == "Costco" else random.uniform(0.29, 0.31)
        price = cost / (1 - target_margin)
    price = round(price, 2)
    return {"sku": sku, "qty": qty, "unit_price": price, "discount_amt": round(price * qty * random.uniform(0, 0.02), 2), "promo_id": promo}


for _ in range(5000):
    cust = random.choice(customers)
    days_ago = random.randint(0, 89)
    day = as_of - timedelta(days=days_ago)
    rdd = day + timedelta(days=random.randint(1, 4))
    if days_ago <= 2:
        status = random.choices(["OPEN", "CANCELLED"], weights=[92, 8])[0]
        rdd = max(rdd, as_of)
    else:
        status = random.choices(["SHIPPED", "INVOICED", "CANCELLED"], weights=[25, 67, 8])[0]
    late_rate = 0.35 if cust["region"] == "QC" else 0.01
    if 3 <= days_ago <= 10 and random.random() < late_rate:
        status, rdd = "OPEN", day + timedelta(days=random.randint(1, days_ago - 1))
    add_order(cust, day, status, rdd, [make_line(cust, day) for _ in range(random.randint(3, 5))])

by_id = {c["ship_to_cust"]: c for c in customers}
for cid in hotspots:
    for day, hour in [(as_of - timedelta(days=1), 9), (as_of, 7), (as_of, 8)]:
        sku = random.choice(skus_by_brand["ACE Bakery"])
        price = round(product[sku]["standard_cost"] * random.uniform(0.80, 0.95), 2)
        qty = random.randint(10, 16)
        item = {"sku": sku, "qty": qty, "unit_price": price, "discount_amt": round(price * qty * 0.01, 2), "promo_id": ACE}
        add_order(by_id[cid], day, "OPEN", as_of + timedelta(days=1), [item], hour)

loblaw = [c for c in customers if c["retailer"] == "Loblaw"]
this_monday = as_of - timedelta(days=as_of.weekday())
for weeks_ago in range(0, 12):
    factor = {0: 0.30, 1: 0.40, 2: 0.60, 3: 0.80}.get(weeks_ago, 1.0)
    monday = this_monday - timedelta(weeks=weeks_ago)
    for _ in range(20):
        day = min(monday + timedelta(days=random.randint(0, 6)), as_of)
        lp = product[decline_sku]["list_price"]
        qty = max(1, round(40 * factor * random.uniform(0.97, 1.03)))
        item = {"sku": decline_sku, "qty": qty, "unit_price": round(lp * 0.95, 2), "discount_amt": 0.0, "promo_id": None}
        status = "OPEN" if day >= as_of - timedelta(days=2) else "INVOICED"
        add_order(random.choice(loblaw), day, status, max(day + timedelta(days=2), as_of), [item])

print(len(headers), "orders", len(lines), "lines", "hotspots:", hotspots, "declining sku:", decline_sku)


In [0]:
budget = []
for back in range(3, -1, -1):
    month = month_start(as_of, back)
    for brand in BRANDS:
        for retailer in RETAILERS:
            revenue, units = actuals.get((brand, retailer, month), [0.0, 0])
            factor = random.uniform(0.96, 1.04)
            if back == 0 and brand == "D'Italiano" and retailer == "Sobeys":
                factor = 1 / 0.90
            if back == 0 and brand == "D'Italiano" and retailer == "Metro":
                factor = 1 / 1.08
            budget.append({
                "brand": brand,
                "retailer": retailer,
                "month": month,
                "budget_revenue": round(max(revenue, 1000.0) * factor, 2),
                "budget_units": int(max(units, 100) * factor),
            })

random.shuffle(headers)
for folder, rows in [
    ("products", products),
    ("customers", customers),
    ("promotions", promotions),
    ("budget", budget),
    ("sales_order_header", headers),
    ("sales_order_lines", lines),
]:
    write_json_lines(folder, rows)
    print(f"{folder}: {len(rows)} rows")


In [0]:
from databricks.sdk import WorkspaceClient

if dbutils.widgets.get("build_pipeline") == "true":
    w = WorkspaceClient()
    name = f"{team}-sales-orders"
    pipeline = next(p for p in w.pipelines.list_pipelines(filter=f"name LIKE '%{name}'") if p.name == name)
    update_id = w.pipelines.start_update(pipeline.pipeline_id, full_refresh=True).update_id
    while True:
        state = w.pipelines.get_update(pipeline.pipeline_id, update_id).update.state.value
        if state in ("COMPLETED", "FAILED", "CANCELED"):
            break
        time.sleep(15)
    print(f"{name} update {update_id}: {state}")
    assert state == "COMPLETED", f"Initial pipeline build ended in {state}"
